# 038 — Desalinhamento formação × mercado sobre as previsões

Responde à pergunta central do DemandA: **onde a contratação prevista (admissões, gold de previsões do 037) diverge da oferta de formação profissional (concluintes/matrículas INEP repartidos pela bridge CINE×CBO)**, no grão UF × subgrupo.

## Indicadores

* **Admissões por concluinte** — razão entre admissões (previstas 12m vs. reais 12m anteriores) e concluintes do último Censo disponível repartidos pela bridge. Alta razão = pressão de contratação não acompanhada pela formação local.
* **Tendência (delta)** — razão prevista ÷ razão histórica: o desalinhamento cresce ou ameniza?
* **Admissões por matrícula** — pressão sobre o estoque de formação em andamento.
* **Classificação relativa** — PRESSÃO ALTA / EQUILIBRADO / EXCEDENTE, por tercis da razão nacional (sem limiar absoluto arbitrário).

## Caveats

* Concluinte não é proxy perfeita de oferta de mão de obra (mercado contrata rotatividade, não só recém-formados) — o indicador mede **pressão relativa**, não lei de mercado.
* Formação do Censo 2024 (último disponível; Censo 2025 pendente) repartida de forma UNIFORME pela bridge provisória (decisão documentada no 035).
* Previsão 12m inclui horizontes recursivos h≥2 (degradação documentada no 037).

In [0]:
# Configurações, parâmetros e preflight das fontes
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime

dbutils.widgets.text("tabela_previsoes", "workspace.gold.previsoes_admissoes", "Gold de previsões (037)")
dbutils.widgets.text("versao_previsao", "0.1.0", "Versão do lote de previsões a consumir")
dbutils.widgets.text("tabela_features", "workspace.gold.ml_features_mensal_uf_cbo", "Wide table (histórico real)")
dbutils.widgets.text("tabela_saida", "workspace.gold.desalinhamento_formacao_mercado", "Tabela-alvo da análise")
dbutils.widgets.text("versao_analise", "0.1.0", "Versão desta análise")
dbutils.widgets.text("fato_inep", "workspace.gold.fato_inep_educacao_uf_anual", "Fato INEP anual")
dbutils.widgets.text("tabela_bridge", "workspace.gold.bridge_cine_cbo_provisoria", "Bridge CINE×CBO")
dbutils.widgets.text("ano_censo", "2024", "Censo da formação (último disponível)")

TABELA_PREVISOES = dbutils.widgets.get("tabela_previsoes")
VERSAO_PREVISAO = dbutils.widgets.get("versao_previsao")
TABELA_FEATURES = dbutils.widgets.get("tabela_features")
TABELA_SAIDA = dbutils.widgets.get("tabela_saida")
VERSAO_ANALISE = dbutils.widgets.get("versao_analise")
FATO_INEP = dbutils.widgets.get("fato_inep")
BRIDGE = dbutils.widgets.get("tabela_bridge")
ANO_CENSO = int(dbutils.widgets.get("ano_censo"))
DT_PROCESSAMENTO = datetime.now()

# Preflight: o Censo pedido é o último disponível?
ano_inep_max = spark.table(FATO_INEP).agg(F.max("ano_censo")).first()[0]
assert ANO_CENSO <= ano_inep_max, f"Censo {ANO_CENSO} indisponível (máximo: {ano_inep_max})"
print(f"Formação: Censo {ANO_CENSO} (último disponível: {ano_inep_max}) | Previsões: {TABELA_PREVISOES} v{VERSAO_PREVISAO}")

In [0]:
# Formação (concluintes e matrículas) do Censo escolhido, repartida pela bridge — MESMA regra do 035:
# contribuição uniforme por área CINE (área ÷ nº de subgrupos do par), somada por subgrupo.
KEYS = ["sigla_uf", "codigo_cbo_subgrupo"]

df_pares = (
    spark.table(BRIDGE)
    .select("codigo_cine_area_especifica", "codigo_cbo_subgrupo")
    .distinct()
)
w_area = Window.partitionBy("codigo_cine_area_especifica")
df_pares_rep = df_pares.withColumn("n_subgrupos_area", F.count("*").over(w_area))

df_formacao = (
    spark.table(FATO_INEP)
    .where(F.col("ano_censo") == ANO_CENSO)
    .groupBy("sigla_uf", "codigo_cine_area_especifica")
    .agg(
        F.sum("concluintes").alias("concluintes_area"),
        F.sum("matriculas").alias("matriculas_area"),
    )
    .join(df_pares_rep, "codigo_cine_area_especifica", "inner")
    .withColumn("concluintes_rep", F.col("concluintes_area") / F.col("n_subgrupos_area"))
    .withColumn("matriculas_rep", F.col("matriculas_area") / F.col("n_subgrupos_area"))
    .groupBy(*KEYS)
    .agg(
        F.round(F.sum("concluintes_rep"), 2).alias("concluintes_censo"),
        F.round(F.sum("matriculas_rep"), 2).alias("matriculas_censo"),
    )
)

qtd_series_formacao = df_formacao.count()
total_concluintes = df_formacao.agg(F.sum("concluintes_censo")).first()[0]
print(f"Formação {ANO_CENSO} repartida: {qtd_series_formacao} séries | "
      f"{total_concluintes:,.0f} concluintes (soma repartida)")
# 702 = produto cheio 27×26 (a formação cobre TODOS os pares da bridge); o painel CAGED tem 698
# — 4 pares sem movimentação em todo o período. O join a jusante é movido pelas previsões (698).
assert qtd_series_formacao >= 698, f"Formação cobre menos séries que o painel: {qtd_series_formacao}"

In [0]:
# Previsões 12m (lote vigente) e os 12 meses REAIS anteriores — janelas consecutivas para leitura de tendência
df_prev_base = spark.table(TABELA_PREVISOES).where(F.col("versao_previsao") == VERSAO_PREVISAO)
ultimo_mes_real = df_prev_base.agg(F.max("ultimo_mes_real")).first()[0]
inicio_janela_real = F.add_months(F.lit(ultimo_mes_real), -12)

df_prev_12m = df_prev_base.groupBy(*KEYS).agg(
    F.round(F.sum("previsao_admissoes"), 1).alias("previsao_admissoes_12m"),
)
df_real_12m = (
    spark.table(TABELA_FEATURES)
    .where((F.col("versao_transformacao") == "0.1.0")
           & (F.col("competencia_mov") > inicio_janela_real)
           & (F.col("competencia_mov") <= F.lit(ultimo_mes_real)))
    .groupBy(*KEYS)
    .agg(F.sum("admissoes").alias("admissoes_reais_12m"))
)
print(f"Previsão: 12m a partir de {ultimo_mes_real:%Y-%m} | Histórico real: janela consecutiva anterior")

# Indicadores de desalinhamento
df_des = (
    df_prev_12m
    .join(df_real_12m, KEYS, "left")
    .join(df_formacao, KEYS, "left")
    .withColumn("adm_por_concluinte_prev", F.round(
        F.when(F.col("concluintes_censo") > 0, F.col("previsao_admissoes_12m") / F.col("concluintes_censo")), 2))
    .withColumn("adm_por_concluinte_hist", F.round(
        F.when(F.col("concluintes_censo") > 0, F.col("admissoes_reais_12m") / F.col("concluintes_censo")), 2))
    .withColumn("adm_por_matricula_prev", F.round(
        F.when(F.col("matriculas_censo") > 0, F.col("previsao_admissoes_12m") / F.col("matriculas_censo")), 2))
    # Tendência: razão prevista ÷ razão histórica (>1 = pressão crescendo)
    .withColumn("tendencia_razao", F.round(
        F.when(F.col("adm_por_concluinte_hist") > 0,
               F.col("adm_por_concluinte_prev") / F.col("adm_por_concluinte_hist")), 3))
)

# Classificação relativa à mediana nacional (tercis — sem limiar absoluto arbitrário)
p25 = df_des.agg(F.percentile_approx("adm_por_concluinte_prev", 0.25)).first()[0]
p75 = df_des.agg(F.percentile_approx("adm_por_concluinte_prev", 0.75)).first()[0]
df_des = df_des.withColumn("classificacao", F
    .when(F.col("adm_por_concluinte_prev") >= p75, "PRESSAO_ALTA")
    .when(F.col("adm_por_concluinte_prev") <= p25, "EXCEDENTE_FORMACAO")
    .otherwise("EQUILIBRADO"))

print(f"Tercis da razão nacional (admissões por concluinte, 12m previstos): p25 = {p25:.1f} | p75 = {p75:.1f}")
display(df_des.groupBy("classificacao").agg(F.count("*").alias("series"), F.round(F.sum("previsao_admissoes_12m"), 0).alias("prev_12m")))

In [0]:
# Gravação idempotente — append por versao_analise
df_saida = df_des.select(
    *KEYS, "previsao_admissoes_12m", "admissoes_reais_12m", "concluintes_censo", "matriculas_censo",
    "adm_por_concluinte_prev", "adm_por_concluinte_hist", "adm_por_matricula_prev",
    "tendencia_razao", "classificacao",
    F.lit(str(ANO_CENSO)).alias("ano_censo_formacao"),
    F.lit(VERSAO_PREVISAO).alias("versao_previsao_origem"),
    F.lit(VERSAO_ANALISE).alias("versao_analise"),
    F.lit(DT_PROCESSAMENTO).alias("dt_processamento"),
)

ja_existe = 0
try:
    ja_existe = spark.table(TABELA_SAIDA).where(F.col("versao_analise") == VERSAO_ANALISE).count()
except Exception:
    ja_existe = 0

if ja_existe > 0:
    print(f"Versão {VERSAO_ANALISE} já gravada ({ja_existe} linhas) — append ignorado (idempotente).")
else:
    df_saida.write.mode("append").saveAsTable(TABELA_SAIDA)
    print(f"Gravado: {df_saida.count()} linhas em {TABELA_SAIDA} (v{VERSAO_ANALISE}).")

# Validação de leitura de volta
chk = spark.table(TABELA_SAIDA).where(F.col("versao_analise") == VERSAO_ANALISE).agg(
    F.count("*").alias("linhas"),
    F.countDistinct("sigla_uf", "codigo_cbo_subgrupo").alias("series"),
    F.sum(F.when(F.col("previsao_admissoes_12m").isNull(), 1).otherwise(0)).alias("prev_nulas"),
).first()
assert chk["linhas"] == 698 and chk["series"] == 698 and chk["prev_nulas"] == 0, f"Validação falhou: {chk}"
print(f"Gold lida de volta: {chk['linhas']} linhas | {chk['series']} séries | previsões nulas: {chk['prev_nulas']}")

In [0]:
# Resumos analíticos — onde está o desalinhamento
print("=== TOP 15 séries com MAIOR pressão (admissões por concluinte, 12m previstos) ===")
display(
    spark.table(TABELA_SAIDA).where(F.col("versao_analise") == VERSAO_ANALISE)
    .orderBy(F.desc("adm_por_concluinte_prev"))
    .limit(15)
)

print("=== Agregado por UF (razão = soma prevista ÷ soma concluintes) ===")
display(
    spark.table(TABELA_SAIDA).where(F.col("versao_analise") == VERSAO_ANALISE)
    .groupBy("sigla_uf")
    .agg(
        F.round(F.sum("previsao_admissoes_12m"), 0).alias("previsao_12m"),
        F.round(F.sum("concluintes_censo"), 0).alias("concluintes"),
        F.round(F.sum("previsao_admissoes_12m") / F.sum("concluintes_censo"), 1).alias("adm_por_concluinte"),
        F.round(F.avg("tendencia_razao"), 3).alias("tendencia_media"),
        F.sum(F.when(F.col("classificacao") == "PRESSAO_ALTA", 1).otherwise(0)).alias("series_pressao_alta"),
    )
    .orderBy(F.desc("adm_por_concluinte"))
)

print("=== Agregado por subgrupo (nacional) — maiores razões ===")
display(
    spark.table(TABELA_SAIDA).where(F.col("versao_analise") == VERSAO_ANALISE)
    .groupBy("codigo_cbo_subgrupo")
    .agg(
        F.round(F.sum("previsao_admissoes_12m"), 0).alias("previsao_12m"),
        F.round(F.sum("concluintes_censo"), 0).alias("concluintes"),
        F.round(F.sum("previsao_admissoes_12m") / F.sum("concluintes_censo"), 1).alias("adm_por_concluinte"),
    )
    .orderBy(F.desc("adm_por_concluinte"))
)

## Observações e limitações

* **Pressão relativa, não lei de mercado** — a razão admissões/concluintes mistura rotatividade com entrada de recém-formados; valores altos indicam contratação intensa frente à formação local, não que "cada vaga ficará sem profissional".
* **Repartição uniforme da bridge provisória** — mesma decisão do 035; quando a bridge definitiva (MEC) chegar, os concluintes repartidos podem mudar e a análise deve ser refeita (bumpar `versao_analise`).
* **Censo 2024 como formação corrente** — o Censo 2025 ainda não foi publicado; a formação "de hoje" é a de 2024 até lá.
* **Janelas consecutivas** — previsão 2026-09..2027-08 vs. real 2025-09..2026-08; a tendência compara períodos equivalentes no tempo.
* **Classificação em tercis da razão nacional** (p25/p75) — evita limiares absolutos arbitrários; rótulos relativos ao próprio painel de 698 séries.
* **Consumo** — dashboards leem `workspace.gold.desalinhamento_formacao_mercado` filtrando `versao_analise` vigente; cada re-execução com nova versão de previsões gera nova versão da análise.